# Step 3: controlled comparison of ByteTrack's second association

Research question: with identical YOLOX detections and other tracker settings fixed, what changes when low-confidence second association is disabled?

**full_bytetrack** uses the official tracker. **high_only** uses a separate copy with one assignment changed: the low-confidence candidate mask is empty. Both receive the same cached post-NMS detections on every frame. This is an ablation of ByteTrack, not a SORT implementation and not a proposed new algorithm.

Use a Colab GPU and run cells in order. All seven MOT17 half-validation scenes are selected. This notebook can reuse /content/MOT17.zip and the official checkpoint from the earlier notebook if your runtime is still connected. A new runtime downloads the approximately 5.5 GB archive again. No Google Drive mount is required.

The notebook includes source-diff checks, synthetic behavior checks, standard TrackEval metrics, comparison against your previous baseline, and downloadable evidence including the detector cache. It has been syntax-checked locally but has not yet been run on a Colab GPU.

Sources: [official ByteTrack code](https://github.com/FoundationVision/ByteTrack), [paper](https://arxiv.org/abs/2110.06864), [MOT17](https://motchallenge.net/data/MOT17/), [TrackEval](https://github.com/JonathonLuiten/TrackEval). This is an AI-assisted learning notebook. Follow the course's rules on assessed work.


## 1. Check the GPU
If this fails, select a GPU runtime and rerun. Keep Colab's matching PyTorch and torchvision packages.


In [ ]:
import sys, subprocess, json, hashlib, re, os, time, shutil
from pathlib import Path
import torch, torchvision
assert torch.cuda.is_available(), 'Choose a GPU under Runtime > Change runtime type.'
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Install demo dependencies
The old repository requirements include dated optional ONNX packages. This installs the dependencies for the PyTorch video demo instead. It does not change the tracking algorithm or replace PyTorch. If Colab requests a restart, restart and rerun from Step 1.


In [ ]:
packages = ['loguru', 'scikit-image', 'tqdm', 'Pillow', 'thop', 'ninja', 'tabulate',
            'tensorboard', 'lap', 'motmetrics', 'filterpy', 'h5py', 'pycocotools',
            'cython_bbox', 'gdown', 'pandas', 'matplotlib', 'scipy']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)


## 3. Get the official source and record compatibility edits
The source revision is fixed. Running from the source directory avoids building the optional accelerated COCO evaluator, which this demo does not use. Deprecated `np.float` aliases become `float`, the GUI key wait is disabled, and checkpoint loading explicitly uses PyTorch's restricted weights loader. All edits are saved in a patch file.


In [ ]:
REVISION = 'd1bf0191adff59bc8fcfeaa0b33d3d1642552a99'
REPO = Path('/content/ByteTrack_paper_starter')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/FoundationVision/ByteTrack.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', REVISION], check=True)
for path in (REPO / 'yolox/tracker').glob('*.py'):
    original = path.read_text()
    patched = re.sub(r'np\.float\b', 'float', original)
    if patched != original:
        path.write_text(patched)
demo = REPO / 'tools/demo_track.py'
source = demo.read_text()
source = source.replace('ch = cv2.waitKey(1)', 'ch = -1  # Colab has no GUI keyboard window')
source = source.replace('torch.load(ckpt_file, map_location="cpu")',
                        'torch.load(ckpt_file, map_location="cpu", weights_only=True)')
demo.write_text(source)
OUT = Path('/content/bytetrack_paper_first_run')
OUT.mkdir(exist_ok=True)
patch = subprocess.check_output(['git', '-C', str(REPO), 'diff'], text=True)
(OUT / 'compatibility.patch').write_text(patch)
print(patch)
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO) + os.pathsep + env.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-c',
    'from yolox.exp import get_exp; from yolox.tracker.byte_tracker import BYTETracker; print("Official imports OK")'],
    cwd=REPO, env=env, check=True)


## 4. Download the official ablation checkpoint
The file ID below comes from the official README's ablation model link. It is paired with `yolox_x_ablation.py`. The checkpoint is several hundred MB. If Google Drive rejects the automated download, use the README link to download it manually and upload it into the printed destination using Colab's Files panel, then rerun this cell.

Use only the official checkpoint. A restricted checkpoint-loading error should be inspected rather than bypassed with an unknown file. This model's documented protocol is MOT17 half-training plus CrowdHuman, followed by MOT17 half-validation. This notebook does not download or evaluate that dataset yet.


In [ ]:
import gdown
WEIGHTS = REPO / 'pretrained/bytetrack_ablation.pth.tar'
WEIGHTS.parent.mkdir(exist_ok=True)
if not WEIGHTS.exists():
    downloaded = gdown.download(id='1iqhM-6V_r1FpOlOzrdP_Ejshgk0DxOob', output=str(WEIGHTS), quiet=False)
    if downloaded is None:
        raise RuntimeError(f'Download failed. Manually upload the official checkpoint to {WEIGHTS}.')
assert WEIGHTS.is_file() and WEIGHTS.stat().st_size > 10_000_000, f'Incomplete checkpoint: {WEIGHTS}'
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
weight_hash = sha256_file(WEIGHTS)
print('Checkpoint:', WEIGHTS, 'bytes:', WEIGHTS.stat().st_size)
print('SHA256:', weight_hash, '(recorded for repeatability, not a publisher-verified checksum)')


## 5. Choose the evaluation scenes and download MOT17
Keep ALL_SEQUENCES=True for the seven-scene comparison. Do not tune settings after seeing evaluation results and then describe those results as an untouched test.

The original repository's converter selects only FRCNN-named copies for private-detector evaluation. These are scene names: this notebook still runs its own YOLOX detector, not the provided FRCNN detections. DPM/FRCNN/SDP variants must not be counted as three independent videos.

If the automatic download fails, use the official MOT17 page to download MOT17.zip, then place the file at /content/MOT17.zip and rerun. The archive URL is the official page's download link.


In [ ]:
import urllib.request, zipfile, configparser, csv
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
ALL_SEQUENCES = True #@param {type:"boolean"}
scene_numbers = ['02','04','05','09','10','11','13'] if ALL_SEQUENCES else ['09']
SEQUENCES = [f'MOT17-{x}-FRCNN' for x in scene_numbers]
WORK = Path('/content/bytetrack_official_controlled')
WORK.mkdir(exist_ok=True)
shutil.copyfile(OUT / 'compatibility.patch', WORK / 'bytetrack_compatibility.patch')
ARCHIVE = Path('/content/MOT17.zip')
if not ARCHIVE.is_file():
    partial = ARCHIVE.with_suffix('.zip.part')
    last_percent = [-10]
    def progress(block, size, total):
        if total > 0:
            percent = min(100, int(block * size * 100 / total))
            if percent >= last_percent[0] + 10:
                print(f'MOT17 download: {percent}%')
                last_percent[0] = percent
    urllib.request.urlretrieve('https://motchallenge.net/data/MOT17.zip', str(partial), reporthook=progress)
    assert zipfile.is_zipfile(partial), 'Download was not a ZIP. Check the official download page.'
    partial.rename(ARCHIVE)
assert zipfile.is_zipfile(ARCHIVE), 'Invalid archive; replace it with the official MOT17.zip.'
print('Selected:', SEQUENCES)
print('Archive size, GB:', round(ARCHIVE.stat().st_size / 1e9, 2))


## 6. Extract the exact validation range and preserve ground truth
The official converter uses zero-based validation indices N//2+1 through N-1. Therefore the first original one-based frame is **N//2+2**. For MOT17-09 (525 frames), this means original frames **264–525**, or **262 validation frames**.

We reset tracker state at validation start. Ground truth and predictions both use new one-based validation frame indices. All ground-truth classes and ignore flags remain in the files for standard MOT17 preprocessing. No video compression or coordinate scaling occurs.


In [ ]:
info_by_seq = {}
with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    for seq in SEQUENCES:
        candidates = [n for n in names if n.endswith(f'train/{seq}/seqinfo.ini')]
        assert len(candidates) == 1, f'Cannot uniquely find {seq}/seqinfo.ini in the official archive.'
        ini_name = candidates[0]
        prefix = ini_name[:-len('seqinfo.ini')]
        cfg = configparser.ConfigParser()
        cfg.read_string(archive.read(ini_name).decode('utf-8-sig'))
        s = cfg['Sequence']
        n, w, h, fps = int(s['seqLength']), int(s['imWidth']), int(s['imHeight']), int(s['frameRate'])
        first_frame = n // 2 + 2
        count = n - first_frame + 1
        folder = WORK / 'images' / seq
        folder.mkdir(parents=True, exist_ok=True)
        for original_frame in range(first_frame, n + 1):
            member = prefix + f'img1/{original_frame:06d}.jpg'
            target = folder / f'{original_frame:06d}.jpg'
            if not target.is_file():
                target.write_bytes(archive.read(member))
        full_gt = np.loadtxt(archive.open(prefix + 'gt/gt.txt'), delimiter=',', ndmin=2)
        assert full_gt.shape[1] >= 9
        gt = full_gt[(full_gt[:,0] >= first_frame) & (full_gt[:,0] <= n)].copy()
        gt[:,0] -= first_frame - 1
        assert len(gt) and gt[:,0].min() >= 1 and gt[:,0].max() <= count
        gt_folder = WORK / 'gt' / seq
        (gt_folder / 'gt').mkdir(parents=True, exist_ok=True)
        np.savetxt(gt_folder / 'gt/gt.txt', gt, delimiter=',', fmt='%.8f')
        cfg['Sequence']['seqLength'] = str(count)
        with (gt_folder / 'seqinfo.ini').open('w') as f: cfg.write(f)
        info_by_seq[seq] = dict(original_frames=n, first_original_frame=first_frame,
                                validation_frames=count, width=w, height=h, fps=fps,
                                images=str(folder), gt_rows=len(gt))
(WORK / 'split_info.json').write_text(json.dumps(info_by_seq, indent=2))
display(pd.DataFrame(info_by_seq).T)


## 7. Inspect annotation alignment before scoring
The green boxes below are scored pedestrian ground-truth boxes. Check that they enclose the correct people on the selected first, middle and last validation frames. If boxes or frames do not match, stop and inspect the input. The full ignore/distractor annotations are retained for evaluation, although they are not drawn here.


In [ ]:
fig, axes = plt.subplots(len(SEQUENCES), 3, figsize=(18, 4 * len(SEQUENCES)), squeeze=False)
for row, seq in enumerate(SEQUENCES):
    info = info_by_seq[seq]
    gt = np.loadtxt(WORK / 'gt' / seq / 'gt/gt.txt', delimiter=',', ndmin=2)
    for col, frame_id in enumerate([1, (info['validation_frames']+1)//2, info['validation_frames']]):
        original = info['first_original_frame'] + frame_id - 1
        image = cv2.imread(str(Path(info['images']) / f'{original:06d}.jpg'))
        assert image is not None and image.shape[:2] == (info['height'], info['width'])
        for r in gt[(gt[:,0]==frame_id) & (gt[:,6]==1) & (gt[:,7]==1)]:
            x,y,w,h = r[2:6]
            cv2.rectangle(image,(round(x),round(y)),(round(x+w),round(y+h)),(0,255,0),2)
        axes[row,col].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[row,col].set_title(f'{seq}: original {original}, validation {frame_id}')
        axes[row,col].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
ALIGNMENT_CHECKED = False #@param {type:"boolean"}
assert ALIGNMENT_CHECKED, 'Inspect the green annotation boxes above, then check ALIGNMENT_CHECKED and rerun.'


## 8. Define and check the single ablation
The high_only tracker changes only `inds_second`, making the second-stage detection set empty. The second-stage code can still execute; this is an accuracy ablation, not a claim to remove all second-stage computational overhead. First-stage candidates, matching costs, thresholds, new-track handling and retention logic are unchanged in code. Later track states may diverge because that is the intended consequence of the intervention.

Track IDs are reset before every independent sequence/variant. Each update receives a fresh copy because the official tracker rescales bounding boxes in place.


In [ ]:
import difflib, importlib
from types import SimpleNamespace
sys.path.insert(0, str(REPO))
original_path = REPO / 'yolox/tracker/byte_tracker.py'
original_source = original_path.read_text()
needle = 'inds_second = np.logical_and(inds_low, inds_high)'
replacement = 'inds_second = np.zeros_like(scores, dtype=bool)'
assert original_source.count(needle) == 1, 'Source changed. Inspect before patching.'
ablated_source = original_source.replace(needle, replacement)
assert ablated_source.replace(replacement, needle) == original_source
ablated_path = REPO / 'yolox/tracker/byte_tracker_high_only.py'
ablated_path.write_text(ablated_source)
patch = ''.join(difflib.unified_diff(original_source.splitlines(True), ablated_source.splitlines(True),
                                    fromfile='byte_tracker.py', tofile='byte_tracker_high_only.py'))
(WORK / 'ablation_only.patch').write_text(patch)
print(patch)
importlib.invalidate_caches()
from yolox.tracker.byte_tracker import BYTETracker
from yolox.tracker.byte_tracker_high_only import BYTETracker as HighOnlyTracker
from yolox.tracker.basetrack import BaseTrack
VARIANTS = ['full_bytetrack', 'high_only']

def settings_for(seq):
    buffer = 14 if seq == 'MOT17-05-FRCNN' else (25 if seq == 'MOT17-13-FRCNN' else 30)
    return dict(track_thresh=0.6, track_buffer=buffer, match_thresh=0.9, mot20=False)

def new_tracker(variant, seq):
    BaseTrack._count = 0
    cls = BYTETracker if variant == 'full_bytetrack' else HighOnlyTracker
    assert variant in VARIANTS
    return cls(SimpleNamespace(**settings_for(seq)))

def toy_run(variant, scores):
    tracker = new_tracker(variant, 'MOT17-09-FRCNN')
    observations = []
    for score in scores:
        det = np.array([[10,10,30,60,score]],dtype=np.float32) if score is not None else np.empty((0,5),np.float32)
        tracks = tracker.update(det.copy(), [100,100], [100,100])
        observations.append([(t.track_id,*t.tlwh.tolist(),float(t.score)) for t in tracks])
    assert tracker.frame_id == len(scores), 'Every frame must advance the tracker.'
    return observations

# Weak detections extend a confirmed track only in full ByteTrack.
full = toy_run('full_bytetrack',[0.9,0.9,0.3])
high = toy_run('high_only',[0.9,0.9,0.3])
assert len(full[-1]) == 1 and len(high[-1]) == 0
# Low-confidence detections alone must not initialize a track.
assert toy_run('full_bytetrack',[0.3])[0] == toy_run('high_only',[0.3])[0] == []
# Without weak detections both variants must agree, including an empty frame.
a = toy_run('full_bytetrack',[0.9,0.9,None,0.9])
b = toy_run('high_only',[0.9,0.9,None,0.9])
assert a == b
(WORK / 'control_checks.json').write_text(json.dumps(dict(
    single_assignment_change=True, weak_detection_recovery_check=True,
    no_low_confidence_initialization=True, high_only_input_equivalence=True),indent=2))
print('Control checks passed. Detector and all other settings will remain fixed.')


## 9. Cache YOLOX detections once
This uses the same checkpoint, original Predictor, FP16, confidence cutoff 0.01, NMS 0.7 and 800×1440 model input as your baseline. Cached detections keep the original post-NMS tensor values and dtype. Score multiplication and coordinate rescaling remain inside the original tracker.

The cache signature includes hashes of the JPEG frames, checkpoint, source revision and compatibility edits, detector settings, and package versions. A matching cache is reused on rerun. Both variants subsequently read this exact cache. The model runs in FP16, while the official postprocessing promotes its seven-column outputs to float32 when adding class IDs. Cache those outputs without recasting them. No confidence threshold is raised at the detector to implement the ablation.


In [ ]:
from yolox.exp import get_exp
from yolox.utils import fuse_model
from yolox.tracking_utils.timer import Timer
from tools.demo_track import Predictor
CACHE = WORK / 'cache'
CACHE.mkdir(exist_ok=True)
cache_paths = {}
cache_info = {}
model = None
predictor = None

def input_digest(seq, info):
    h = hashlib.sha256()
    for original in range(info['first_original_frame'], info['original_frames'] + 1):
        path = Path(info['images']) / f'{original:06d}.jpg'
        h.update(path.name.encode())
        h.update(bytes.fromhex(sha256_file(path)))
    return h.hexdigest()

for seq in SEQUENCES:
    info = info_by_seq[seq]
    path = CACHE / f'{seq}.npz'
    signature = dict(sequence=seq, split=info, image_sha256=input_digest(seq,info),
                     checkpoint_sha256=weight_hash, source_revision=REVISION,
                     compatibility_sha256=hashlib.sha256((WORK/'bytetrack_compatibility.patch').read_bytes()).hexdigest(),
                     torch=torch.__version__, torchvision=torchvision.__version__, numpy=np.__version__,
                     test_size=[800,1440], detector_confidence=0.01, nms=0.7, fp16=True, fused=True)
    signature_json = json.dumps(signature,sort_keys=True)
    reuse = False
    if path.exists():
        with np.load(path,allow_pickle=False) as saved:
            reuse = str(saved['signature'].item()) == signature_json
    if not reuse:
        if predictor is None:
            exp = get_exp(str(REPO/'exps/example/mot/yolox_x_ablation.py'),None)
            exp.test_conf, exp.nmsthre = 0.01, 0.7
            assert list(exp.test_size) == [800,1440]
            model = exp.get_model()
            checkpoint = torch.load(WEIGHTS,map_location='cpu',weights_only=True)
            model.load_state_dict(checkpoint['model'])
            del checkpoint
            model = fuse_model(model).eval().cuda().half()
            predictor = Predictor(model,exp,device=torch.device('cuda'),fp16=True)
            with torch.no_grad():
                model(torch.zeros((1,3,*exp.test_size),device='cuda',dtype=torch.float16))
            torch.cuda.synchronize()
        parts, offsets = [], [0]
        timer = Timer()
        torch.cuda.synchronize()
        started = time.perf_counter()
        for frame_id in range(1,info['validation_frames']+1):
            original = info['first_original_frame'] + frame_id - 1
            image = cv2.imread(str(Path(info['images'])/f'{original:06d}.jpg'))
            assert image is not None and image.shape[:2] == (info['height'],info['width'])
            output, _ = predictor.inference(image,timer)
            data = output[0].detach().cpu().numpy().copy() if output[0] is not None else np.empty((0,7),np.float32)
            assert data.ndim == 2 and data.shape[1] == 7 and data.dtype == np.float32
            parts.append(data)
            offsets.append(offsets[-1]+len(data))
            timer.toc()
            if frame_id % 100 == 0: print(seq,'detector:',frame_id,'/',info['validation_frames'])
        torch.cuda.synchronize()
        elapsed = time.perf_counter()-started
        detections = np.concatenate(parts,axis=0)
        np.savez_compressed(path,boxes=detections,offsets=np.asarray(offsets,dtype=np.int64),
                            signature=signature_json,detector_wall_seconds=elapsed)
    with np.load(path,allow_pickle=False) as saved:
        boxes, offsets = saved['boxes'], saved['offsets']
        assert len(offsets) == info['validation_frames']+1 and offsets[0] == 0 and offsets[-1] == len(boxes)
        assert np.all(offsets[1:] >= offsets[:-1]) and np.isfinite(boxes).all()
        scores = boxes[:,4] * boxes[:,5]
        weak = int(((scores>0.1)&(scores<0.6)).sum())
        cache_info[seq] = dict(cache_sha256=sha256_file(path), boxes=len(boxes), weak_candidates=weak,
                               reused=reuse, detector_wall_seconds=float(saved['detector_wall_seconds']))
    cache_paths[seq] = path
    print(seq, 'cache reused' if reuse else 'cache created', '| weak candidates:',weak)
assert sum(i['weak_candidates'] for i in cache_info.values()) > 0, 'No weak detections; verify detector threshold.'
(WORK/'cache_info.json').write_text(json.dumps(cache_info,indent=2))
# Free GPU model memory. Tracking uses only cached detections from here onward.
del predictor, model
torch.cuda.empty_cache()


## 10. Replay identical caches with both trackers
Each variant starts from a fresh tracker and uses the same per-scene arguments, frame order and output filters (minimum area 100, maximum width/height 1.6). Each sees the raw cached detections, with its own copy to avoid in-place rescaling corrupting the cache. Empty frames advance both trackers. Cache hashes are checked again afterward.

Tracking call times are recorded for context, but the empty-mask ablation is not a clean computational-speed study. Focus the conclusions on accuracy and association metrics.


In [ ]:
run_info = {}
for seq in SEQUENCES:
    info = info_by_seq[seq]
    with np.load(cache_paths[seq],allow_pickle=False) as saved:
        boxes, offsets = saved['boxes'].copy(), saved['offsets'].copy()
    before = hashlib.sha256(boxes.tobytes()).hexdigest()
    for variant in VARIANTS:
        tracker = new_tracker(variant,seq)
        assert vars(tracker.args) == settings_for(seq)
        output = WORK/'trackers'/variant/'data'/f'{seq}.txt'
        output.parent.mkdir(parents=True,exist_ok=True)
        processed, written, tracker_seconds = 0, 0, 0.0
        with output.open('w',newline='') as handle:
            writer = csv.writer(handle)
            for frame_id in range(1,info['validation_frames']+1):
                # Preserve the seven-column post-NMS detector output and its original dtype.
                frame_data = torch.from_numpy(boxes[offsets[frame_id-1]:offsets[frame_id]].copy())
                start = time.perf_counter()
                tracks = tracker.update(frame_data,[info['height'],info['width']],[800,1440])
                tracker_seconds += time.perf_counter()-start
                for track in tracks:
                    x,y,w,h = track.tlwh
                    if w*h > 100 and h > 0 and w/h <= 1.6:
                        writer.writerow([frame_id,track.track_id,f'{x:.2f}',f'{y:.2f}',f'{w:.2f}',f'{h:.2f}',f'{track.score:.2f}',-1,-1,-1])
                        written += 1
                processed += 1
        assert processed == info['validation_frames'] == tracker.frame_id
        assert hashlib.sha256(boxes.tobytes()).hexdigest() == before
        assert sha256_file(cache_paths[seq]) == cache_info[seq]['cache_sha256']
        run_info[f'{seq}/{variant}'] = dict(frames=processed,boxes_written=written,
            tracker_args=settings_for(seq),cache_sha256=cache_info[seq]['cache_sha256'],tracker_seconds=tracker_seconds)
        print(seq,variant,processed,'frames;',written,'tracked boxes')
(WORK/'run_info.json').write_text(json.dumps(run_info,indent=2))


## 11. Evaluate both variants with the same TrackEval configuration
The standard MOT17 ignore/distractor preprocessing remains enabled for both variants. The evaluator combines sequences using metric-specific rules, not a simple average. Scores are percentages and error counts are integers.


In [ ]:
EVAL_REVISION = '12c8791b303e0a0b50f753af204249e622d0281a'
EVAL_REPO = Path('/content/TrackEval_paper_step2')
if not (EVAL_REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/JonathonLuiten/TrackEval.git',str(EVAL_REPO)],check=True)
subprocess.run(['git','-C',str(EVAL_REPO),'checkout',EVAL_REVISION],check=True)
for path in (EVAL_REPO / 'trackeval').rglob('*.py'):
    source = path.read_text()
    patched = re.sub(r'np\.(float|int|bool)\b', lambda m: m.group(1), source)
    if source != patched: path.write_text(patched)
(WORK / 'trackeval_compatibility.patch').write_text(subprocess.check_output(['git','-C',str(EVAL_REPO),'diff'],text=True))
sys.path.insert(0,str(EVAL_REPO))
import trackeval
assert ALIGNMENT_CHECKED
cfg = trackeval.Evaluator.get_default_eval_config()
cfg.update(USE_PARALLEL=False, PRINT_RESULTS=True, PRINT_ONLY_COMBINED=False,
           DISPLAY_LESS_PROGRESS=True, TIME_PROGRESS=False, PLOT_CURVES=False, PRINT_CONFIG=False)
data_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
data_cfg.update(GT_FOLDER=str(WORK/'gt'), TRACKERS_FOLDER=str(WORK/'trackers'),
                OUTPUT_FOLDER=str(WORK/'metrics'), TRACKERS_TO_EVAL=VARIANTS,
                BENCHMARK='MOT17', SPLIT_TO_EVAL='train', SKIP_SPLIT_FOL=True,
                SEQ_INFO={s:i['validation_frames'] for s,i in info_by_seq.items()},
                DO_PREPROC=True, PRINT_CONFIG=False)
dataset = trackeval.datasets.MotChallenge2DBox(data_cfg)
results, messages = trackeval.Evaluator(cfg).evaluate([dataset],
    [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()])
rows = []
for variant in VARIANTS:
    for seq in SEQUENCES + ['COMBINED_SEQ']:
        r = results[dataset.get_name()][variant][seq]['pedestrian']
        rows.append(dict(sequence=seq,variant=variant,MOTA=100*float(r['CLEAR']['MOTA']),
            IDF1=100*float(r['Identity']['IDF1']),HOTA=100*float(np.mean(r['HOTA']['HOTA'])),
            IDSW=int(r['CLEAR']['IDSW']),FP=int(r['CLEAR']['CLR_FP']),FN=int(r['CLEAR']['CLR_FN'])))
summary = pd.DataFrame(rows)
summary.to_csv(WORK/'comparison.csv',index=False)
display(summary.round(3))
measures = ['MOTA','IDF1','HOTA','IDSW','FP','FN']
full = summary[summary.variant=='full_bytetrack'].set_index('sequence')[measures]
high = summary[summary.variant=='high_only'].set_index('sequence')[measures]
delta = full - high
delta.to_csv(WORK/'full_minus_high_only.csv')
print('Full ByteTrack minus high-only: positive score differences and negative error-count differences favor full ByteTrack.')
display(delta.round(3))


## 12. Check the full variant against your previous baseline
The reference below contains your previous seven-scene scores from ByteTrack_official_MOT17_halfval_results-2.zip. It is used only for a consistency check, not as an extra experiment or as data for tuning.

The new full variant should closely reproduce that baseline. GPU or library differences can alter floating-point outputs. If scores or counts differ, save the comparison and investigate before attributing those changes to second association. The controlled conclusion comes from the two variants run together here.


In [ ]:
REFERENCE = [{'sequence': 'MOT17-02-FRCNN', 'MOTA': '55.516194331983804', 'IDF1': '58.77531904929165', 'HOTA': '47.62554324977874', 'IDSW': '57', 'FP': '830', 'FN': '3508'}, {'sequence': 'MOT17-04-FRCNN', 'MOTA': '89.68483745553809', 'IDF1': '91.81690347765861', 'HOTA': '79.70901309176601', 'IDSW': '19', 'FP': '883', 'FN': '1592'}, {'sequence': 'MOT17-05-FRCNN', 'MOTA': '76.61602621388144', 'IDF1': '76.1645589883144', 'HOTA': '60.928990192409536', 'IDSW': '28', 'FP': '145', 'FN': '612'}, {'sequence': 'MOT17-09-FRCNN', 'MOTA': '83.049670024314', 'IDF1': '77.8657616892911', 'HOTA': '65.46385977606921', 'IDSW': '14', 'FP': '10', 'FN': '464'}, {'sequence': 'MOT17-10-FRCNN', 'MOTA': '71.58534526422422', 'IDF1': '69.23004429365753', 'HOTA': '54.9197972722434', 'IDSW': '26', 'FP': '211', 'FN': '1446'}, {'sequence': 'MOT17-11-FRCNN', 'MOTA': '70.44498560991809', 'IDF1': '71.0398893244178', 'HOTA': '62.35065111462135', 'IDSW': '11', 'FP': '482', 'FN': '842'}, {'sequence': 'MOT17-13-FRCNN', 'MOTA': '77.31305449936629', 'IDF1': '84.1264327891629', 'HOTA': '65.16638467771206', 'IDSW': '18', 'FP': '72', 'FN': '626'}, {'sequence': 'COMBINED_SEQ', 'MOTA': '77.92540359992579', 'IDF1': '79.96999694047749', 'HOTA': '67.94477469485794', 'IDSW': '173', 'FP': '2633', 'FN': '9090'}]
reference = pd.DataFrame(REFERENCE).set_index('sequence')[measures].astype(float)
common = full.index.intersection(reference.index)
baseline_delta = full.loc[common] - reference.loc[common]
baseline_delta.to_csv(WORK/'full_minus_previous_baseline.csv')
display(baseline_delta.round(6))
exact = bool(np.allclose(baseline_delta.to_numpy(),0,atol=1e-6,rtol=0))
checks = dict(full_matches_previous_scores=exact,shared_detector_cache=True,
              identical_non_ablation_settings=True,all_frames_advanced=True,cache_unchanged=True)
(WORK/'comparison_checks.json').write_text(json.dumps(checks,indent=2))
if exact:
    print('The full variant reproduces the previous baseline scores.')
else:
    print('Full variant differs from the previous baseline. Preserve the results and inspect environment, cache dtype, model and wrapper before interpreting differences.')


## 13. Save the complete comparison archive
The ZIP includes both tracker predictions, evaluator output, metric differences, consistency checks, cache signatures and detector caches. It excludes images, full MOT17.zip and model weights. Download before Colab disconnects. The saved code changes and environment are part of the experiment record.


In [ ]:
metadata = dict(byte_revision=REVISION,trackeval_revision=EVAL_REVISION,
    checkpoint_sha256=weight_hash,python=sys.version,torch=torch.__version__,
    torchvision=torchvision.__version__,numpy=np.__version__,gpu=torch.cuda.get_device_name(0),
    variants=VARIANTS,sequences=SEQUENCES,
    intervention='Only inds_second assignment changed to an all-False mask in high_only',
    shared_settings={s:settings_for(s) for s in SEQUENCES},
    split_rule='Original one-based frames N//2+2 through N; reset tracker at start',
    limitations=['Validation ablation, not unseen leaderboard test performance',
                 'Official modules with documented wrapper and TrackEval',
                 'Both variants advance on empty frames',
                 'Accuracy ablation, not complete computational removal of second-stage code'])
(WORK/'environment.json').write_text(json.dumps(metadata,indent=2))
(WORK/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
(WORK/'high_only_source.py').write_text(ablated_source)
archive_path = Path('/content/ByteTrack_official_controlled_comparison.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in WORK.rglob('*'):
        if path.is_file() and 'images' not in path.relative_to(WORK).parts:
            archive.write(path,path.relative_to(WORK))
from google.colab import files
print('Download:',archive_path,'size MB:',round(archive_path.stat().st_size/1e6,1))
files.download(str(archive_path))


## Read the results before writing conclusions

Send the downloaded comparison ZIP for review. The main table is comparison.csv. full_minus_high_only.csv reports the paired change on the same inputs. For MOTA, IDF1 and HOTA, report differences in percentage points. For IDSW, FP and FN, report count changes.

Do not assume the full variant wins every metric or every scene. Fewer switches alone can reflect missing difficult tracks, so examine FN and identity scores together. Aggregate benefits do not establish improvement in every condition. Any later tuning on these seven scenes is validation/development work.

Keep the model, confidence thresholds and data split fixed for this first comparison. Do not modify the earlier YOLO11 notebooks to substitute for this official-YOLOX ablation.
